In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
import pandas as pd
import torch
from scipy.interpolate import interp1d

# --- CLASS & MODEL IMPORTS ---
from classy import Class 
from classy_altered.classy import Class as Class_altered 
from padé import combined_full_model 
from NN_no_PCA import DRMD_Emulator

# --- 1. MATPLOTLIB SETTINGS ---
plt.rcParams.update(plt.rcParamsDefault)
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.0,
    "lines.linewidth": 2.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "xtick.top": True,
    "ytick.right": True,
})

# --- 2. CONFIGURATION & PATHS ---
N_grid = 512
input_dir_2048 = r"/home/storgaard/OneDrive/concept/grendel_output/2048/"
input_dir_256  = r"/home/storgaard/OneDrive/concept/grendel_output/256/"
emu_checkpoint_path = "/home/storgaard/OneDrive/Speciale/Emulator/top_10_ensembles/emulator_#1.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

k_nyq_2048 = np.pi / (2048 / N_grid)
k_nyq_256  = np.pi / (256 / N_grid)
k_transition = 0.1 * k_nyq_2048
k_max_small = 0.75 * k_nyq_256

baseline_fidm = 0.0285
baseline_dNeff = 0.83
step_dNeff = 0.231 
step_fidm = 0.0076

In [2]:
# --- 3. EMULATOR SETUP ---
print("Loading Neural Network Emulator...")
# FIX: Added weights_only=False to bypass the PyTorch 2.6 security restriction
checkpoint = torch.load(emu_checkpoint_path, map_location=device, weights_only=False)
emu_k_grid = checkpoint['k_grid']
mean_in = checkpoint['input_scaler_mean']
scale_in = checkpoint['input_scaler_scale']

emulator = DRMD_Emulator(target_mean=torch.zeros(25), target_scale=torch.ones(25), input_dim=3, output_dim=25, hidden_nodes=128).to(device)
emulator.load_state_dict(checkpoint['model_state_dict'])
emulator.eval()

def get_emu_C(k_target, z, abs_dNeff, abs_fidm):
    a_val = 1.0 / (1.0 + z)
    raw_inputs = np.array([abs_fidm, abs_dNeff, a_val])
    inputs_scaled = (raw_inputs - mean_in) / scale_in
    inputs_tensor = torch.tensor(inputs_scaled, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred_Ck = emulator(inputs_tensor).squeeze(0).cpu().numpy()
        
    f_interp = interp1d(emu_k_grid, pred_Ck, kind='cubic', bounds_error=False, fill_value="extrapolate")
    return f_interp(k_target)

# --- 4. N-BODY DATA HANDLERS ---
def load_raw_pk(box_dir, model_name, z):
    a = 1 / (1 + z)
    filepath = os.path.join(box_dir, model_name, f"powerspec/powerspec_size512_lpt2_a={a:.3f}")
    return pd.read_csv(filepath, sep=r'\s+', comment='#', header=None, usecols=[0, 1, 3], names=['k', 'modes', 'Pk'])

def get_nbody_pk(model_name, z):
    df_256 = load_raw_pk(input_dir_256, model_name, z)
    df_2048 = load_raw_pk(input_dir_2048, model_name, z)
    large_mask = df_2048['k'] <= k_transition
    small_mask = (df_256['k'] > k_transition) & (df_256['k'] <= k_max_small) 
    return pd.concat([df_2048[large_mask], df_256[small_mask]], ignore_index=True)


# --- 5. PRECOMPUTE GLOBAL LCDM BASELINES ---
all_possible_zs = [0.0, 0.2, 0.4, 1.0, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
z_str_global = ", ".join(str(z) for z in all_possible_zs)

base_class_params = {
    'h': 0.6766, 'omega_b': 0.02242, 'omega_cdm': 0.1192,
    'A_s': 2.107699e-9, 'n_s': 0.9671182, 'output': 'dTk, mPk, vTk', 'lensing': 'no',
    'N_ur': 2.0328, 'N_ncdm': 1, 'deg_ncdm': 1, 'm_ncdm': 0.06, 'T_ncdm': 0.71611,
    'YHe': 0.245, 'P_k_max_1/Mpc': 15.0, 'k_pivot': 0.05,
    'z_pk': z_str_global
}

print("Computing Global LCDM Baselines (This happens only once)...")
lcdm_halofit = Class()
lcdm_halofit.set({**base_class_params, 'non linear': 'halofit'})
lcdm_halofit.compute()

lcdm_hmcode = Class()
lcdm_hmcode.set({**base_class_params, 'non linear': 'hmcode'})
lcdm_hmcode.compute()

lcdm_cache_halofit = {}
lcdm_cache_hmcode = {}
for z in all_possible_zs:
    # FIX: Route to planck_z5 if the redshift is > 3.0 for the cache
    planck_model = "planck_z5" if z > 3.0 else "planck"
    try:
        k_vals = get_nbody_pk(planck_model, z)['k'].values
        lcdm_cache_halofit[z] = np.array([lcdm_halofit.pk(ki, z) for ki in k_vals])
        lcdm_cache_hmcode[z] = np.array([lcdm_hmcode.pk(ki, z) for ki in k_vals])
    except FileNotFoundError:
        print(f"Warning: Could not precompute LCDM for z={z} in folder {planck_model}")

Loading Neural Network Emulator...
Computing Global LCDM Baselines (This happens only once)...


In [3]:
def evaluate_and_plot(test_name, test_config, framework):
    dNeff_shift = test_config["dNeff"]
    fidm_shift = test_config["fidm"]
    z_to_plot = test_config["zs"]
    
    print(f"  -> Generating {framework} plot for {test_name}...")
    
    abs_dNeff = baseline_dNeff + dNeff_shift * step_dNeff
    abs_fidm = baseline_fidm + fidm_shift * step_fidm
    
    z_str_local = ", ".join(str(z) for z in z_to_plot)
    drmd_params = {
        'l_switch_limber': 9, 'z_stop': 10**(4.83), 'G_over_aH_drmd_ini': 10**(13.057),
        'delta_Neff_drmd': abs_dNeff, 'f_idm_drmd': abs_fidm, 'z_pk': z_str_local
    }
    
    drmd_cosmo = None
    if framework == "HMcode":
        drmd_cosmo = Class()
        drmd_cosmo.set({**base_class_params, **drmd_params, 'non linear': 'hmcode'})
        drmd_cosmo.compute()
    elif framework == "Unaltered":
        drmd_cosmo = Class()
        drmd_cosmo.set({**base_class_params, **drmd_params, 'non linear': 'halofit'})
        drmd_cosmo.compute()
    elif framework == "ABM":
        drmd_cosmo = Class_altered()
        drmd_cosmo.set({**base_class_params, **drmd_params, 'non linear': 'halofit'})
        drmd_cosmo.compute()
        
    # --- LAYOUT UPDATE: 2 Rows, 3 Columns ---
    fig = plt.figure(figsize=(18, 10)) 
    
    # Back to wspace=0.0 globally! Row-sharing allows us to hide inner Y-ticks everywhere.
    gs_outer = gridspec.GridSpec(2, 3, wspace=0.0, hspace=0.0) 
    
    is_test_5 = (test_name == "test_5")
    is_test_6 = (test_name == "test_6")
    
    # Trackers for Standard (Global Sharing)
    ax_main_shared_global = None
    ax_res_shared_global = None
    global_res_min = np.inf
    global_res_max = -np.inf
    
    # Trackers for Test 5 (Row-wise Sharing)
    ax_main_shared_row = [None, None]
    ax_res_shared_row = [None, None]
    row_res_min = [np.inf, np.inf]
    row_res_max = [-np.inf, -np.inf]
    
    for i, z in enumerate(z_to_plot):
        row = i // 3 
        col = i % 3  
        
        gs_inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_outer[i], 
                                                    height_ratios=[3, 1], hspace=0.0)
        
        # --- AXES SHARING LOGIC ---
        if is_test_5:
            # Share axes across the specific row
            if ax_main_shared_row[row] is None:
                ax_main = fig.add_subplot(gs_inner[0])
                ax_main_shared_row[row] = ax_main
            else:
                ax_main = fig.add_subplot(gs_inner[0], sharey=ax_main_shared_row[row])
                
            if ax_res_shared_row[row] is None:
                ax_res = fig.add_subplot(gs_inner[1], sharex=ax_main)
                ax_res_shared_row[row] = ax_res
            else:
                ax_res = fig.add_subplot(gs_inner[1], sharex=ax_main, sharey=ax_res_shared_row[row])
        else:
            # Share axes globally across the entire 2x3 grid
            if ax_main_shared_global is None:
                ax_main = fig.add_subplot(gs_inner[0])
                ax_main_shared_global = ax_main
            else:
                ax_main = fig.add_subplot(gs_inner[0], sharey=ax_main_shared_global)

            if ax_res_shared_global is None:
                ax_res = fig.add_subplot(gs_inner[1], sharex=ax_main)
                ax_res_shared_global = ax_res
            else:
                ax_res = fig.add_subplot(gs_inner[1], sharex=ax_main, sharey=ax_res_shared_global)

        planck_model = "planck_z5" if is_test_5 else "planck"
        
        # Fetch N-body Data
        df_nbody_lcdm = get_nbody_pk(planck_model, z)
        df_nbody_drmd = get_nbody_pk(test_name, z)
        k_vals = df_nbody_lcdm['k'].values 
        Pk_nbody_lcdm = df_nbody_lcdm['Pk'].values
        Pk_nbody_drmd = df_nbody_drmd['Pk'].values
        
        # Calculate Model P(k)
        if framework == "HMcode":
            Pk_lcdm_base = lcdm_cache_hmcode[z]
            Pk_drmd_model = np.array([drmd_cosmo.pk(ki, z) for ki in k_vals])
        elif framework in ["Unaltered", "ABM"]:
            Pk_lcdm_base = lcdm_cache_halofit[z]
            Pk_drmd_model = np.array([drmd_cosmo.pk(ki, z) for ki in k_vals])
        elif framework == "Pade":
            Pk_lcdm_base = lcdm_cache_halofit[z]
            C_k = combined_full_model(k_vals, z, dNeff_shift, fidm_shift)['C']
            Pk_drmd_model = Pk_lcdm_base * C_k
        elif framework == "Emulator":
            Pk_lcdm_base = lcdm_cache_halofit[z]
            C_k = get_emu_C(k_vals, z, abs_dNeff, abs_fidm)
            Pk_drmd_model = Pk_lcdm_base * C_k

        # Calculate Ratios
        ratio_lcdm = Pk_nbody_lcdm / Pk_lcdm_base
        ratio_drmd = Pk_nbody_drmd / Pk_drmd_model
        double_ratio = ratio_drmd / ratio_lcdm
        
        # Track limits for scaling
        if is_test_5:
            row_res_min[row] = min(row_res_min[row], np.nanmin(double_ratio))
            row_res_max[row] = max(row_res_max[row], np.nanmax(double_ratio))
        else:
            global_res_min = min(global_res_min, np.nanmin(double_ratio))
            global_res_max = max(global_res_max, np.nanmax(double_ratio))
        
        # --- Plot Top Panel ---
        # Your exact styling: markersize=8
        ax_main.semilogx(k_vals, ratio_lcdm, label=r'$\Lambda$CDM', color='tab:blue', marker='o', markersize=8, ls='None')
        ax_main.semilogx(k_vals, ratio_drmd, label=r'DRMD', color='tab:orange', marker='x', markersize=8, ls='None')
        ax_main.axhline(1.0, color='grey', linestyle='--')
        ax_main.text(0.05, 0.83, f"$z = {z:.1f}$", transform=ax_main.transAxes, verticalalignment='center')
        
        if row == 0 and col == 0:
            ax_main.legend(frameon=False, loc='best', fontsize='18')
            names = {
                "test_1": "Test 1: dNeff=+1.75, fidm=-1.75",
                "test_3": "Test 2: dNeff=-2.0, fidm=+0.75",
                "test_4": "Test 3: dNeff=+2.5, fidm=-2.5",
                "test_5": "Test 4: Baseline at high-z",
                "test_6": "Test 5: dNeff=+3.5, fidm=+3.5",
                "test_7": "Test 6: SIDR model, no fidm"
            }
            
            # Your exact styling: fontsize=20
            ax_main.text(0.45, 0.93, names.get(test_name, test_name), transform=ax_main.transAxes, 
                         verticalalignment='center', horizontalalignment='center', 
                        color='k', fontname='Serif', fontsize=20)
            
        if is_test_6 and framework == "ABM":
            # Make the main axis ylims different for this specific case to better show the trends
            ax_main.set_ylim(0.81, 1.22)

        if is_test_6 and framework == "Pade":
            # Make the main axis ylims different for this specific case to better show the trends
            ax_main.set_ylim(0.81, 1.22)

        # --- Plot Bottom Panel ---
        # Your exact styling: markersize=8
        ax_res.semilogx(k_vals, double_ratio, color='tab:green', marker='s', markersize=8, ls='None')
        ax_res.axhline(1.0, color='black', lw=1, ls='-')
        
        # --- FORMATTING & TICK HIDING ---
        ax_main.set_xlim(2e-3, 6)
        ax_res.set_xlim(2e-3, 6)
        
        # Always hide main X-axis labels (residual is directly beneath it)
        ax_main.tick_params(labelbottom=False)
        
        # Hide top row residual X-axis labels
        if row == 0:
            ax_res.tick_params(labelbottom=False)
            
        # Hide middle and right column Y-axis labels globally! 
        # (Since we are row-sharing or global-sharing, cols 1 & 2 never need Y-labels)
        if col > 0:
            ax_main.tick_params(labelleft=False)
            ax_res.tick_params(labelleft=False)
        
        # Only add labels to the outer edges of the entire grid
        if col == 0:
            ax_main.set_ylabel(r'$P_{\mathrm{Nbody}} / P_{\mathrm{Model}}$')
            ax_res.set_ylabel(r'$\mathcal{D}(k,z)$')
        
        if row == 1: 
            ax_res.set_xlabel(r'$k \ [1/\mathrm{Mpc}]$')

    # --- APPLY CALCULATED LIMITS ---
    if is_test_5:
        # Apply the independent bounds for row 0 and row 1
        for r in range(2):
            if ax_res_shared_row[r] is not None:
                ax_res_shared_row[r].set_ylim(row_res_min[r] - 0.01, row_res_max[r] + 0.01)
    else:
        # Apply the global bounds to the shared residual Y-axis for standard tests
        if ax_res_shared_global is not None:
            ax_res_shared_global.set_ylim(global_res_min - 0.01, global_res_max + 0.01)

    plt.tight_layout()
    
    output_dir = "evaluation_plots_new"
    os.makedirs(output_dir, exist_ok=True)
    
    save_name = f"Ratio_Grid_{framework}_{test_name}.pdf"
    save_path = os.path.join(output_dir, save_name)
    
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) 
    
    if drmd_cosmo is not None:
        drmd_cosmo.struct_cleanup()
        drmd_cosmo.empty()

In [4]:
# ==========================================
# 7. EXECUTE THE MASTER LOOP
# ==========================================
default_zs = [0.0, 0.2, 0.4, 1.0, 2.0, 3.0]
test_cases = {
    # "test_1": {"dNeff": 1.75, "fidm": -1.75, "zs": default_zs},
    # "test_3": {"dNeff": -2.0, "fidm": 0.75, "zs": default_zs},
    # "test_4": {"dNeff": 2.5, "fidm": -2.5, "zs": default_zs},
    # "test_5": {"dNeff": 0.0, "fidm": 0.0, "zs": [2.5, 3.0, 3.5, 4.0, 4.5, 5.0]},
    "test_6": {"dNeff": 3.5, "fidm": 3.5, "zs": default_zs},
    # "test_7": {"dNeff": 0.0, "fidm": -3.75, "zs": default_zs}
}

frameworks = ["Unaltered", "HMcode", "ABM", "Pade", "Emulator"]

print("\nStarting Master Evaluation Loop...")
for test_folder, test_config in test_cases.items():
    print(f"\nEvaluating {test_folder}:")
    for fw in frameworks:
        evaluate_and_plot(
            test_name=test_folder, 
            test_config=test_config, 
            framework=fw
        )

print("\nAll evaluations complete and PDFs saved!")


Starting Master Evaluation Loop...

Evaluating test_6:
  -> Generating Unaltered plot for test_6...
  -> Generating HMcode plot for test_6...
  -> Generating ABM plot for test_6...
  -> Generating Pade plot for test_6...
  -> Generating Emulator plot for test_6...

All evaluations complete and PDFs saved!
